# Öğrenci 2 — Eksik ölçümler, uç değerler ve veri kalitesi

Hücreleri **yukarıdan aşağı sırayla** çalıştır (Shift+Enter). Toplam süre Colab'da ~10 dakika.

**Ön koşul:** Drive'da *Benimle paylaşılanlar* → `ebv_data` → sağ tık → **Düzenle > Drive'ıma kısayol ekle**.

## 1. Drive'ı bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DRIVE = '/content/drive/MyDrive/ebv_data'
assert os.path.exists(DATA_DRIVE), 'ebv_data bulunamadı: Drive\'ıma kısayol ekle adımını yaptın mı?'
print(os.listdir(DATA_DRIVE))

## 2. Repoyu indir ve kütüphaneleri kur

In [ ]:
%cd /content
!rm -rf elektrik-buyuk-veri
!git clone https://github.com/sumeyr4yuce/elektrik-buyuk-veri.git
%cd /content/elektrik-buyuk-veri
# PR henüz birleştirilmediyse Öğrenci 2'nin branch'ine geç; birleştirildiyse main'de kal
!git checkout ogrenci2-veri-kalitesi 2>/dev/null && echo 'branch: ogrenci2-veri-kalitesi' || echo 'branch: main'
!pip install -q pyspark==3.5.3 pyarrow matplotlib
!ls src/ogrenci2_veri_kalitesi

## 3. Veriyi Colab diskine kopyala
Drive'dan okumak yavaş ve süre ölçümlerini bozar. Kopyalama ortak veriyi **değiştirmez**.

In [ ]:
!mkdir -p /content/data_local
!cp -r {DATA_DRIVE}/processed /content/data_local/
!cp -r {DATA_DRIVE}/subsets /content/data_local/
!du -sh /content/data_local/*

## 4. Hızlı deneme: 100K alt küme (~1 dk)

In [ ]:
!python -m src.ogrenci2_veri_kalitesi.s05_data_quality --data /content/data_local --input-subset 100k --out /content/out_100k --no-write

## 5. Asıl çalıştırma: verinin tamamı + 100K/1M/5M benchmark (~6-8 dk)

In [ ]:
!EBV_DRIVER_MEMORY=8g python -m src.ogrenci2_veri_kalitesi.s05_data_quality --data /content/data_local --out /content/outputs/ogrenci2 --benchmark

## 6. Raporu gör

In [ ]:
from IPython.display import Markdown, display, Image
import glob
print(open('/content/outputs/ogrenci2/docs/ogrenci2_bulgular.md', encoding='utf-8').read()[:3000])
for f in sorted(glob.glob('/content/outputs/ogrenci2/docs/figures/*.png')):
    print(f); display(Image(f))

## 7. Temiz veriyi kontrol et

In [ ]:
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.config('spark.sql.session.timeZone','UTC').getOrCreate()
df = spark.read.parquet('/content/outputs/ogrenci2/clean/readings_clean.parquet')
df.printSchema()
df.groupBy('quality_flag').count().orderBy(F.desc('count')).show(15, truncate=False)
print('model için kullanılabilir:', df.where('use_for_model').count(), '/', df.count())
# ACORN kontrolü: boş (null) değil, ACORN-A / Affluent gibi değerler görmelisin
df.groupBy('acorn_grouped').agg(F.countDistinct('meter_id').alias('abone')).show()

## 8. Çıktıları Drive'a kaydet (ekiple paylaşmak için)
`ebv_data`'nın **içine değil**, kendi klasörüne kaydediyoruz. Sonra bu klasörü ekiple paylaş.

In [ ]:
# Eski temiz veriyi önce sil: Spark dosya adları her çalıştırmada değişir,
# silmeden kopyalarsak eski ve yeni dosyalar yan yana kalır ve satırlar iki kez sayılır.
!rm -rf /content/drive/MyDrive/ogrenci2_cikti/clean
!mkdir -p /content/drive/MyDrive/ogrenci2_cikti
!cp -r /content/outputs/ogrenci2/* /content/drive/MyDrive/ogrenci2_cikti/
!ls /content/drive/MyDrive/ogrenci2_cikti